In [9]:
!pip install transformers torch scikit-learn datasets -q

In [2]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset

In [8]:
print(torch.cuda.is_available())  # must print True
print(torch.cuda.get_device_name(0))  # prints your GPU name

True
Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/PolyGuard/01-data/train.csv')
print("Loaded rows:", len(df))
print(df['label'].value_counts())

Mounted at /content/drive
Loaded rows: 21802
label
False    11819
True      9983
Name: count, dtype: int64


In [4]:
# convert True/False to 1/0
df['label'] = df['label'].astype(int)

# split into train and test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))

Training samples: 17441
Testing samples: 4361


In [5]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
print("Tokenizer loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Tokenizer loaded!


In [6]:
def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

train_encodings = tokenize(train_df['code'])
test_encodings  = tokenize(test_df['code'])

print("Tokenization done!")

Tokenization done!


In [7]:
class CodeDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = CodeDataset(train_encodings, train_df['label'].reset_index(drop=True))
test_dataset  = CodeDataset(test_encodings,  test_df['label'].reset_index(drop=True))

print("Datasets ready!")

Datasets ready!


In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/codebert-base",
    num_labels=2
)
print("Model loaded!")

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded!


In [17]:
import os
os.makedirs('/content/drive/MyDrive/PolyGuard/03-models/checkpoints', exist_ok=True)

training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/PolyGuard/03-models/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_steps=500,
    load_best_model_at_end=True,
    # Removed deprecated logging_dir
    TENSORBOARD_LOGGING_DIR='/content/drive/MyDrive/PolyGuard/04-results/logs',
    logging_steps=100,
    warmup_steps=100,
    weight_decay=0.01,
)
print("Training settings ready!")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training settings ready!


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

print("Starting training... this will take 20-40 mins with GPU")
trainer.train()
print("Training complete!")

Starting training... this will take 20-40 mins with GPU


Epoch,Training Loss,Validation Loss
1,0.702312,0.690084
2,0.695191,0.689858
3,0.687800,0.688076


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Training complete!


In [14]:
class CodeDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = CodeDataset(train_encodings, train_df['label'].reset_index(drop=True))
test_dataset  = CodeDataset(test_encodings,  test_df['label'].reset_index(drop=True))

print("Datasets ready!")

Datasets ready!


In [15]:
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/codebert-base",
    num_labels=2
)
print("Model loaded!")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded!


In [18]:
import os
os.makedirs('/content/drive/MyDrive/PolyGuard/03-models/checkpoints', exist_ok=True)

training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/PolyGuard/03-models/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_steps=500,
    load_best_model_at_end=True,
    # Removed deprecated logging_dir
    TENSORBOARD_LOGGING_DIR='/content/drive/MyDrive/PolyGuard/04-results/logs',
    logging_steps=100,
    warmup_steps=100,
    weight_decay=0.01,
)
print("Training settings ready!")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training settings ready!


In [20]:
os.makedirs('/content/drive/MyDrive/PolyGuard/03-models/polyguard_model', exist_ok=True)

model.save_pretrained('/content/drive/MyDrive/PolyGuard/03-models/polyguard_model')
tokenizer.save_pretrained('/content/drive/MyDrive/PolyGuard/03-models/polyguard_model')

print("Model saved to Drive! Everyone can now load it.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to Drive! Everyone can now load it.
